In [1]:
import torch
import torch.nn.functional as F
import datasets
import huggingface_hub
import matplotlib.font_manager as font_manager
import pandas as pd
import matplotlib.pyplot as plt
import transformers
import seaborn as sns
from datasets import load_dataset
from datasets import Dataset
import torch.nn as nn
from transformers import AutoModelForSequenceClassification,Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score

In [2]:
data=load_dataset("dair-ai/emotion")

In [3]:
data

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

In [4]:
train=data["train"]
test=data["test"]
val=data["validation"]

In [5]:
from transformers import AutoTokenizer
model_checkpoint="distilbert-base-uncased"
tokenizer =AutoTokenizer.from_pretrained(model_checkpoint)

In [6]:
def token(i):
    return tokenizer(i["text"],padding=True,truncation=True)

token_trian=train.map(token, batched=True,batch_size=None).remove_columns(["text"])
token_test=test.map(token,batched=True,batch_size=None).remove_columns(["text"])
token_val=val.map(token,batched=True,batch_size=None).remove_columns(["text"])


Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [7]:
from transformers import AutoModel
model_checkpoint = "distilbert-base-uncased"
device = torch.device("cuda")
model = AutoModel.from_pretrained(model_checkpoint).to(device)

model=AutoModelForSequenceClassification.from_pretrained(model_checkpoint,num_labels=6).to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
def print_parmeter_with_classfier(model):
    all_parm=0
    train_parm=0
    for name,parm in model.named_parameters():
        if "classifier" in name:
            parm.requires_grad=True
        elif parm.ndim==1:
            parm.requires_grad=True
        else:
            parm.requires_grad=False
        all_parm+=parm.numel()
        if parm.requires_grad ==True:
            train_parm+=parm.numel()
       
    percentage = (train_parm / all_parm) * 100 if all_parm > 0 else 0
    print(f"all parm:{all_parm} | trian_parm:{train_parm}, present: {percentage}")


In [9]:
def print_parmeter_without_classfier(model):
    all_parm=0
    tarin_parm=0
    for _,parm in model.named_parameters():
        if parm.ndim==1:
            parm.requires_grad=True
        else:
            parm.requires_grad=False

        all_parm+=parm.numel()
        if parm.requires_grad==True:
            tarin_parm+=parm.numel()

    percentage = (tarin_parm / all_parm) * 100 if all_parm > 0 else 0
    print(f"all parm:{all_parm} | trian_parm:{tarin_parm}, present: {percentage}")

In [10]:
print("with classfier")
print_parmeter_with_classfier(model=model)
print('*'*80)
print("without classfier")
print_parmeter_without_classfier(model=model)

with classfier
all parm:66958086 | trian_parm:656646, present: 0.980682153907446
********************************************************************************
without classfier
all parm:66958086 | trian_parm:62214, present: 0.09291484227909381


In [11]:
for name, module in model.named_modules():
    if 'attn' in name or 'attention' in name:  
        print(name) 

distilbert.transformer.layer.0.attention
distilbert.transformer.layer.0.attention.q_lin
distilbert.transformer.layer.0.attention.k_lin
distilbert.transformer.layer.0.attention.v_lin
distilbert.transformer.layer.0.attention.out_lin
distilbert.transformer.layer.0.attention.dropout
distilbert.transformer.layer.1.attention
distilbert.transformer.layer.1.attention.q_lin
distilbert.transformer.layer.1.attention.k_lin
distilbert.transformer.layer.1.attention.v_lin
distilbert.transformer.layer.1.attention.out_lin
distilbert.transformer.layer.1.attention.dropout
distilbert.transformer.layer.2.attention
distilbert.transformer.layer.2.attention.q_lin
distilbert.transformer.layer.2.attention.k_lin
distilbert.transformer.layer.2.attention.v_lin
distilbert.transformer.layer.2.attention.out_lin
distilbert.transformer.layer.2.attention.dropout
distilbert.transformer.layer.3.attention
distilbert.transformer.layer.3.attention.q_lin
distilbert.transformer.layer.3.attention.k_lin
distilbert.transformer.la

In [12]:
from peft import LoraConfig, get_peft_model

config =LoraConfig(
    r=16, 
    lora_alpha=32,
    target_modules=["q_lin", "v_lin"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS"
)

In [13]:
model=get_peft_model(model,config)

In [14]:
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

class CastOutputToFloat(nn.Sequential):
    def forward(self, x):
        return super().forward(x).to(torch.float32)

model.classifier = CastOutputToFloat(model.classifier)

In [15]:
args=TrainingArguments(
    output_dir="./lora_results",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    learning_rate=3e-4)
def eval(pred):
    logits, labels = pred
    preds = logits.argmax(axis=-1)
    f1 = f1_score(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1}


In [16]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=token_trian, 
    eval_dataset=token_val,   
    compute_metrics=eval,  
)

trainer.train()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.601963,0.265424,0.903500,0.904524
2,0.249926,0.189673,0.925000,0.924814
3,0.181751,0.168832,0.932500,0.932647


c:\Users\nice\anaconda3\envs\tourch\Lib\site-packages\peft\utils\other.py:1419: UserWarning: Unable to fetch remote file due to the following error [Errno 11001] getaddrinfo failed - silently ignoring the lookup for the file config.json in distilbert-base-uncased.
  warnings.warn(
c:\Users\nice\anaconda3\envs\tourch\Lib\site-packages\peft\utils\save_and_load.py:372: UserWarning: Could not find a config file in distilbert-base-uncased - will assume that the vocabulary was not modified.
  warnings.warn(


TrainOutput(global_step=1500, training_loss=0.34454662577311196, metrics={'train_runtime': 421.2222, 'train_samples_per_second': 113.954, 'train_steps_per_second': 3.561, 'total_flos': 1102817089152000.0, 'train_loss': 0.34454662577311196, 'epoch': 3.0})

In [20]:
merged_model=model.merge_and_unload()

In [21]:
merged_model.save_pretrained(r"C:\Users\nice\Desktop\Puresoft\Pure_Soft-emotion-poject\app\final_model_lora")
tokenizer.save_pretrained(r"C:\Users\nice\Desktop\Puresoft\Pure_Soft-emotion-poject\app\final_model_lora")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('C:\\Users\\nice\\Desktop\\Puresoft\\Pure_Soft-emotion-poject\\app\\final_model_lora\\tokenizer_config.json',
 'C:\\Users\\nice\\Desktop\\Puresoft\\Pure_Soft-emotion-poject\\app\\final_model_lora\\tokenizer.json')